Import des données depuis Kaggle

In [ ]:
from copy import deepcopy

import kagglehub
import os

path = kagglehub.dataset_download("ashery/chexpert")
print("Chemin local :", path)
for root, dirs, files in os.walk(path):
    for f in files:
        if f.endswith(".csv"):
            print(os.path.join(root, f))

Création d'un DataFrame avec les données (code venant de Kaggle)

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "train.csv"  # ou le chemin relatif exact retourné à l'étape 1

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "ashery/chexpert",
    file_path,
)
print(df.head())

Import des données sous forme d'un DataFrame et premier nettoyage pour avoir **uniquement des valeurs numérique**.
Les colonnes concernées sont ```Sex``` (Male, Female -> 0, 1) et ```Frontal/Latéral``` (Frontal, Latérale -> 0, 1)

In [ ]:
import pandas as pd
import copy

def numeric_df(df):
    new_df = copy.deepcopy(df)

    sexes = {"Male": 0, "Female": 1}
    new_df["Sex"] = new_df["Sex"].map(sexes)

    axes = {"Frontal": 0, "Lateral": 1}
    new_df["Frontal/Lateral"] = new_df["Frontal/Lateral"].map(axes)

    ap_pa_dict = {"AP": 0, "PA": 1}
    new_df["AP/PA"] = new_df["AP/PA"].map(ap_pa_dict)

    return new_df

for root, dirs, files in os.walk("data"):
    for file in files:
        if file.endswith(".csv"):
            df = pd.read_csv(os.path.join(root, file))

            df = numeric_df(df)

            os.makedirs("cleaned", exist_ok=True)
            name_file, ext = os.path.splitext(file)
            df.to_csv(os.path.join("cleaned", f"{name_file}_cleaned{ext}"), index=False)

Premier nettoyage des images ```.jpg``` en recadrant sur le centre avec une **taille de 320x320**.

In [ ]:
from PIL import Image
import numpy as np

SIZE = 320


def crop_img(image):
    img_w, img_h = image.size

    left = 0 + (img_w - SIZE) // 2
    top = 0 + (img_h - SIZE) // 2
    right = left + SIZE
    bottom = top + SIZE

    img_cropped = image.crop((left, top, right, bottom))
    return img_cropped

input_root = "data"
output_root = "cleaned"

for root, dirs, files in os.walk("data"):
    for file in files:
        if file.endswith(".jpg") and not file.startswith("._"):
            img = Image.open(os.path.join(root, file)).convert('L')

            img_cropped = crop_img(img)

            rel_path = os.path.relpath(root, input_root)    # Chemin relatif

            # Recréer les dossiers dans ./cleaned
            output_dir = os.path.join(output_root, rel_path)
            os.makedirs(output_dir, exist_ok=True)

            name_file, ext = os.path.splitext(file)
            img_cropped.save(os.path.join(output_dir, f"{name_file}_cleaned{ext}"))


In [ ]:
def preprocessing(image):
    x = np.array(image, dtype=np.float32)

    # Normaliser (entre 0 et 1)
    x /= 255.0

    # Standardiser (centré réduit)
    x = (x - x.mean()) / x.std()

    # Ajouter la dimension de l'image (grayscale -> 1)
    x = np.expand_dims(x, axis=0)       # (1, 320, 320)

    return x